GUARDRAILS

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API Key: ")


In [22]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

Two Approaches to Guardrails

Deterministic Guardrails

Rule-based: regex, keyword matching, explicit checks
✅ Fast, predictable, cost-effective
❌ May miss nuanced violations


Model-Based Guardrails
Uses LLMs/classifiers for semantic understanding
✅ Catches subtle/nuanced issues
❌ Slower and more expensive

In [23]:
# Quick illustration of the two approaches

import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [25]:
from langchain_groq import ChatGroq

# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


Built-in Guardrail — PII Detection Middleware

LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [41]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I can help with that, but I need to know what specifically you'd like assistance with. For example:

- Looking up your account or order details
- Resolving a billing or payment issue
- Updating account information
- Something else

Also, for your security, please avoid sharing full card numbers or complete email addresses in chat. The last four digits of a card and a general description of the issue are usually enough to get started.

What would you like help with?


In [42]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='66eb958a-92c5-4d71-b000-7d401d4ce495'),
  AIMessage(content="I can help with that, but I need to know what specifically you'd like assistance with. For example:\n\n- Looking up your account or order details\n- Resolving a billing or payment issue\n- Updating account information\n- Something else\n\nAlso, for your security, please avoid sharing full card numbers or complete email addresses in chat. The last four digits of a card and a general description of the issue are usually enough to get started.\n\nWhat would you like help with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 298, 'total_tokens': 397, 'completion_time': 0.191437151, 'completion_tokens_details': None, 'prompt_time': 0.021835754, 'prompt_tokens_details': None, 'queue_time': 0.054029975, 'total_time'

In [43]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [44]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='66eb958a-92c5-4d71-b000-7d401d4ce495'),
  AIMessage(content="I can help with that, but I need to know what specifically you'd like assistance with. For example:\n\n- Looking up your account or order details\n- Resolving a billing or payment issue\n- Updating account information\n- Something else\n\nAlso, for your security, please avoid sharing full card numbers or complete email addresses in chat. The last four digits of a card and a general description of the issue are usually enough to get started.\n\nWhat would you like help with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 298, 'total_tokens': 397, 'completion_time': 0.191437151, 'completion_tokens_details': None, 'prompt_time': 0.021835754, 'prompt_tokens_details': None, 'queue_time': 0.054029975, 'total_time'

Built-in Guardrail — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

In [45]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [46]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='a328e57a-a406-4364-85aa-8d71fe76b0a4'), AIMessage(content="I don't have the Q4 results data, so I'll need that information to compose an accurate email. Could you please provide:\n\n1. **Key Q4 results/figures** (revenue, growth, profits, etc.)\n2. **Any specific points or highlights** you'd like to emphasize\n3. **Tone** (formal, celebratory, neutral, etc.)\n4. **Call to action** (if any) — e.g., a meeting, review, or next steps\n\nOnce you share the details, I'll draft and send the email to team@company.com.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 431, 'total_tokens': 557, 'completion_time': 0.248929188, 'completion_tokens_details': None, 'prompt_time': 0.032419114, 'prompt_tokens_details': None, 'queue_time': 0.062035215, 'total_time': 0

In [47]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I don't have the Q4 results data, so I'll need that information to compose an accurate email. Could you please provide:

1. **Key Q4 results/figures** (revenue, growth, profits, etc.)
2. **Any specific points or highlights** you'd like to emphasize
3. **Tone** (formal, celebratory, neutral, etc.)
4. **Call to action** (if any) — e.g., a meeting, review, or next steps

Once you share the details, I'll draft and send the email to team@company.com.


In [48]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
The deletion was not executed. If you'd like to proceed, let me know and I can retry, or if you'd prefer a different approach (e.g., soft-delete by setting a flag, or reviewing which records would be affected first), let me know.


Custom Guardrail — Before-Agent Hook (Input Filter)

Use before_agent() to validate or block requests before any LLM processing begins.

In [49]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [50]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine learning (ML)** is a branch of artificial intelligence (AI) that focuses on building systems that can learn from data. Instead of being explicitly programmed with rules for every possible scenario, an ML algorithm identifies patterns in data and uses those patterns to make predictions or decisions.

### Key Characteristics:
1. **Data-Driven**: ML models are trained on large datasets. The quality and quantity of data significantly impact the model's performance.
2. **Pattern Recognition**: The core task is to find underlying structures or relationships in the data.
3. **Prediction and Decision-Making**: Once trained, the model can apply what it has learned to new, unseen data to predict outcomes (e.g., classifying an email as spam or not, predicting stock prices, recognizing faces in images).
4. **Improvement Over Time**: In many cases, models can improve their accuracy as they are exposed to more data.

### Common Types of Machine Learning:
- **Superv

In [51]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


Custom Guardrail — After-Agent Hook (Output Safety)

Use after_agent() to validate the final agent response before the user sees it.

In [62]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq  # <-- Use ChatGroq here
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use ChatGroq with qwen or openai/gpt-oss-safeguard-20b
        self.safety_model = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")


Output safety agent created!


In [63]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
I don't have access to real-time data, including live weather information. To check the weather for today, please:

1. Check a local weather app or website (e.g., Weather.com, AccuWeather, or your device's built-in weather app)
2. Search for "weather in [your city]" in a search engine
3. Ask a voice assistant on your device

If you let me know your location, I can suggest reliable sources to check the current conditions!


Layered / Combined Guardrails

In [64]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!
